In [ ]:
# nanoGPT from scratch on Kaggle
# 第 1 个 cell：环境检查、目标说明、导入依赖、固定随机种子
#
# 这份 notebook 的目标不是直接调用官方 nanoGPT 的 train.py，
# 而是在 Kaggle 里一步一步“自己写出一个 nanoGPT”。
# 官方源码已经放在仓库的 nanoGPT-reference/ 目录中，后续每一步都会对照它来实现。
#
# 当前第 1 格只做准备工作：
# 1. 确认 PyTorch 和 GPU 状态
# 2. 处理 Kaggle 上 P100 + 新版 PyTorch 不兼容的问题
# 3. 导入后续会用到的基础库
# 4. 固定随机种子，方便复现实验
# 5. 设置一个 device 变量，后续张量和模型都放到这个设备上

import math
import os
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F


print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


def choose_device():
    """选择当前 Kaggle 环境里真正能安全使用的设备。"""
    if not torch.cuda.is_available():
        print("GPU not found, using CPU. 也能跑小实验，但会慢一些。")
        return "cpu"

    name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    print("GPU:", name)
    print("CUDA capability:", f"sm_{props.major}{props.minor}")

    # Kaggle 有时会分配 Tesla P100。P100 是 sm_60。
    # 你遇到的 PyTorch 2.10.0+cu128 构建只支持 sm_70 及以上。
    # 这种情况下 torch.cuda.is_available() 仍然可能是 True，
    # 但真正执行 CUDA kernel 时会失败，所以这里主动退回 CPU。
    if props.major < 7:
        print("\n注意：当前 GPU 是 P100/sm_60，但这个 PyTorch 构建不支持 sm_60。")
        print("本 notebook 将临时使用 CPU。想用 GPU，请在 Kaggle Accelerator 里优先选择 T4。")
        print("如果 Kaggle 只能给 P100，也可以改装兼容旧架构的 PyTorch，但初学阶段先不折腾环境。\n")
        return "cpu"

    return "cuda"


device = choose_device()


# 固定随机种子。
# 训练神经网络时，初始化权重、打乱数据、采样 token 都会用到随机数。
# 固定 seed 后，每次从头运行 notebook，结果更容易复现和对比。
seed = 1337
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if device == "cuda":
    torch.cuda.manual_seed(seed)


# 允许 TensorFloat-32。它是 NVIDIA GPU 上的一种加速格式。
# 对我们这种学习实验来说，可以让矩阵乘法更快。
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


print("Using device:", device)
print("Seed:", seed)


# 后续我们会按这个顺序继续写：
# Cell 2：按官方路线准备 Tiny Shakespeare，并用 GPT-2 BPE 变成 token
# Cell 3：写 get_batch，构造 x/y 训练样本
# Cell 4：先实现 BigramLanguageModel，理解最小语言模型
# Cell 5：实现单头 causal self-attention
# Cell 6：实现多头注意力
# Cell 7：实现 MLP 和 Block
# Cell 8：组装 GPTConfig 和 GPT
# Cell 9：写训练循环
# Cell 10：写 generate 生成文本


In [ ]:
# 第 2 个 cell：官方 Tiny Shakespeare + GPT-2 BPE 数据准备
#
# 这一格对应 nanoGPT 官方源码：
#   nanoGPT-reference/data/shakespeare/prepare.py
#
# 这是你刚选的路线：
#
#   Tiny Shakespeare -> GPT-2 BPE tokenizer -> train.bin / val.bin
#
# 它和 shakespeare_char 不一样：
#   shakespeare_char 是字符级分词，一个字符一个 token。
#   shakespeare 是 GPT-2 BPE 分词，后面可以接官方 finetune_shakespeare.py。
#
# 官方微调配置文件：
#   nanoGPT-reference/config/finetune_shakespeare.py

import os
import subprocess
import sys
from pathlib import Path

import numpy as np


def ensure_package(import_name, pip_name=None):
    """缺少依赖时自动安装，方便在 Kaggle 新环境中直接运行。"""
    try:
        return __import__(import_name)
    except ModuleNotFoundError:
        package_name = pip_name or import_name
        print(f"当前环境没有 {package_name}，正在安装。")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])
        return __import__(import_name)


requests = ensure_package("requests")
tiktoken = ensure_package("tiktoken")


# 输出目录。
# 官方仓库里是 data/shakespeare/。
# Kaggle 里我们写到 /kaggle/working/data/shakespeare/，这样后面训练脚本能按同样结构找数据。

if Path("/kaggle/working").exists():
    data_dir = Path("/kaggle/working/data/shakespeare")
else:
    data_dir = Path("data/shakespeare")
data_dir.mkdir(parents=True, exist_ok=True)

input_file_path = data_dir / "input.txt"
train_bin_path = data_dir / "train.bin"
val_bin_path = data_dir / "val.bin"

print("数据目录:", data_dir)


# 官方 Tiny Shakespeare 原始数据。
# 只有约 1MB，很适合先完整跑通“准备数据 -> 训练 -> 生成”的闭环。
#
# 如果 Kaggle 网络关闭，也可以把官方 input.txt 放到 Kaggle Input。
# 代码会先尝试找 /kaggle/input/**/input.txt；找不到再下载官方地址。

data_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"


def find_uploaded_input_txt():
    kaggle_input_dir = Path("/kaggle/input")
    if not kaggle_input_dir.exists():
        return None
    matches = sorted(kaggle_input_dir.rglob("input.txt"))
    return matches[0] if matches else None


if input_file_path.exists():
    print("发现已有 input.txt，直接复用:", input_file_path)
else:
    uploaded_input = find_uploaded_input_txt()
    if uploaded_input is not None:
        print("从 Kaggle Input 读取 input.txt:", uploaded_input)
        input_file_path.write_text(uploaded_input.read_text(encoding="utf-8"), encoding="utf-8")
    else:
        print("下载官方 Tiny Shakespeare:", data_url)
        response = requests.get(data_url, timeout=60)
        response.raise_for_status()
        input_file_path.write_text(response.text, encoding="utf-8")


text = input_file_path.read_text(encoding="utf-8")

print("\n原始文本字符数:", f"{len(text):,}")
print("前 500 个字符预览：")
print(text[:500])


# 和官方 data/shakespeare/prepare.py 保持一致：
# 先按原始文本字符数量切成 90% / 10%，再分别做 GPT-2 BPE 编码。
#
# 为什么先切文本再分词？
# 因为 BPE 分词会受边界附近字符影响。官方就是先切 raw text，再 encode。

n = len(text)
train_text = text[:int(n * 0.9)]
val_text = text[int(n * 0.9):]

print("\n训练集原始字符数:", f"{len(train_text):,}")
print("验证集原始字符数:", f"{len(val_text):,}")


# GPT-2 BPE tokenizer。
# BPE 可以理解成“常见文本片段优先合并”的分词方式。
# 它不是一个字符一个 token，而是把常见单词、空格加单词、子词片段变成 token。
#
# 微调官方 GPT-2 权重时，词表大小必须和 GPT-2 一致：50257。
# 注意：从零训练 nanoGPT 时常把 50257 padding 到 50304；
# 但微调 GPT-2 / GPT-2 XL 时，模型原来的词嵌入表就是 50257 行，所以这里用 50257。

enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab

print("\nTokenizer:", enc.name)
print("GPT-2 词表大小 vocab_size:", vocab_size)
# 官方 shakespeare/prepare.py 没有追加特殊结束标记。
# Tiny Shakespeare 在这个例子里被当成一个连续文本流，所以这里严格照官方，不追加。

train_ids = enc.encode_ordinary(train_text)
val_ids = enc.encode_ordinary(val_text)

print("\n训练集 token 数:", f"{len(train_ids):,}")
print("验证集 token 数:", f"{len(val_ids):,}")
print("官方注释参考：train.bin 约 301,966 tokens，val.bin 约 36,059 tokens")


# 看一个很小的 BPE 例子。
# token id 只是词表里的编号，不代表大小或重要程度。

example_text = "First Citizen:"
example_ids = enc.encode_ordinary(example_text)

print("\nBPE 编码/解码小例子：")
print("原始文本:", example_text)
print("编码后 token id:", example_ids)
print("解码回去:", enc.decode(example_ids))
print("每个 token id 对应的文本片段：")
for token_id in example_ids:
    print(f"{token_id:>6} -> {enc.decode([token_id])!r}")


# 写出官方训练脚本能读取的 .bin。
# 为什么是 uint16？
# GPT-2 token id 最大是 50256，小于 65535，刚好可以用 16 位无符号整数保存。

train_array = np.array(train_ids, dtype=np.uint16)
val_array = np.array(val_ids, dtype=np.uint16)

train_array.tofile(train_bin_path)
val_array.tofile(val_bin_path)

print("\n已写出:")
print(train_bin_path, "大小 MB:", f"{train_bin_path.stat().st_size / 1024**2:.2f}")
print(val_bin_path, "大小 MB:", f"{val_bin_path.stat().st_size / 1024**2:.2f}")


# 后面训练时，用 memmap 读取，和 nanoGPT train.py 的思路一致。
# 这个数据很小，直接读进内存也行；但我们先保持和官方训练代码风格一致。

train_data = np.memmap(train_bin_path, dtype=np.uint16, mode="r")
val_data = np.memmap(val_bin_path, dtype=np.uint16, mode="r")

print("\ntrain_data 形状:", train_data.shape)
print("val_data 形状:", val_data.shape)
print("训练集前 40 个 token id:")
print(train_data[:40].tolist())
print("解码前 40 个 token:")
print(enc.decode(train_data[:40].tolist()))


def encode(s):
    """把字符串变成 GPT-2 BPE token id 列表。"""
    return enc.encode_ordinary(s)


def decode(ids):
    """把 GPT-2 BPE token id 列表还原成字符串。"""
    return enc.decode(ids)


# 这一格产生的关键变量：
#   enc / encode / decode       GPT-2 BPE 分词器和辅助函数
#   vocab_size                  GPT-2 词表大小，50257
#   data_dir                    数据目录，对应官方 data/shakespeare
#   train_bin_path / val_bin_path
#   train_data / val_data       np.memmap 形式的 token 序列
#
# 下一格 Cell 3 会讲：
#   如何从 train_data 中随机取一批连续 token，
#   构造 x 和 y，让模型学习“看到前文，预测下一个 token”。
